# Chapter 2: Detecting Hallucinations Before Your Users Do

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RudrenduPaul/hardening-llm-systems-production/blob/main/companion-code/ch02-detecting-hallucinations/ch02_notebook.ipynb)
## Hardening LLM Systems in Production
**Author:** Rudrendu Paul | https://orcid.org/0009-0008-0141-4690

---

### What this notebook builds

This notebook implements the full hallucination detection stack from Chapter 2:

1. **HallucinationMetric**, deepeval wrapper that scores individual responses for faithfulness
2. **RAGASPipeline**, RAGAS faithfulness scoring that decomposes answers into atomic claims
3. **CombinedHallucinationScorer**, weighted ensemble of both metrics (60% deepeval + 40% RAGAS)
4. **Cohen's kappa**, measure inter-rater reliability before trusting any metric as a CI gate
5. **Power analysis**, compute the sample size needed to detect a meaningful change in hallucination rate

### Why two metrics, not one?

deepeval and RAGAS use fundamentally different decomposition strategies.
deepeval compares the full response to each context chunk.
RAGAS breaks the response into atomic claims and checks each claim independently.
Agreement between the two provides stronger evidence of faithfulness than either alone.
Their disagreement is a signal worth investigating.

### Pinned versions

| Package | Version |
|---------|--------|
| deepeval | 0.21.7 |
| ragas | 0.1.21 |
| scikit-learn | >=1.3.0 |
| scipy | >=1.11.0 |

## Manuscript reference

This notebook demonstrates the concepts from Chapter 2 of *Hardening LLM Systems in Production* (Manning, 2026).

| Notebook section | Manuscript listing | Class / function |
|------------------|--------------------|------------------|
| deepeval hallucination scoring | Listing 2.3 | `deepeval HallucinationMetric` |
| RAGAS faithfulness scoring | Listing 2.4 | `RAGAS faithfulness` |
| Ensemble scorer | Listing 2.5 | `CombinedHallucinationScorer` |
| Inter-rater reliability | Listing 2.7 | `compute_cohens_kappa` |
| Sample size / power analysis | Listing 2.8 | `compute_sample_size` |


In [1]:
# ── Colab setup ────────────────────────────────────────────────────────────
# This cell only runs when executed in Google Colab.
# Local Jupyter users: skip — all code is stdlib or pip-installable.
import sys, os

if 'google.colab' in sys.modules:
    !git clone -q https://github.com/RudrenduPaul/hardening-llm-systems-production.git
    os.chdir('hardening-llm-systems-production/companion-code/ch02-detecting-hallucinations')
    !pip install -q deepeval==0.21.7 ragas==0.1.21 matplotlib
    print('Colab setup complete — repo cloned, packages installed.')


## Setup

Install the evaluation stack. The companion scripts include mock scorers
so every cell runs even without an OpenAI API key.

In [2]:
# Install dependencies (comment out if already installed)
# !pip install deepeval==0.21.7 ragas==0.1.21 scikit-learn>=1.5.0,<2.0 scipy>=1.13.0,<2.0 openai>=1.35.0,<2.0


In [3]:
import importlib.util, pathlib, sys

print(f"Python {sys.version}")

# Load the companion script
spec = importlib.util.spec_from_file_location("ch02_scripts", pathlib.Path("ch02_scripts.py"))
ch02 = importlib.util.module_from_spec(spec)
sys.modules['ch02_scripts'] = ch02  # required for Python 3.14
spec.loader.exec_module(ch02)

print(f"deepeval available : {ch02._DEEPEVAL_AVAILABLE}")
print(f"ragas available    : {ch02._RAGAS_AVAILABLE}")
print(f"sklearn available  : {ch02._SKLEARN_AVAILABLE}")
print(f"scipy available    : {ch02._SCIPY_AVAILABLE}")

if not ch02._DEEPEVAL_AVAILABLE:
    print("\n[NOTE] deepeval not installed — mock scorer will be used.")
if not ch02._RAGAS_AVAILABLE:
    print("[NOTE] ragas not installed — mock scorer will be used.")

Python 3.14.3 (main, Feb  3 2026, 15:32:20) [Clang 17.0.0 (clang-1700.6.3.2)]


deepeval available : False
ragas available    : False
sklearn available  : True
scipy available    : True

[NOTE] deepeval not installed — mock scorer will be used.
[NOTE] ragas not installed — mock scorer will be used.


## Section 1: The demo dataset

Four QA pairs, ranging from faithful (Paris is the capital of France)
to clearly hallucinated (Homer's Iliad attributed to Shakespeare).

In [4]:
import json

for i, s in enumerate(ch02.DEMO_SAMPLES, 1):
    print(f"Sample {i}")
    print(f"  Question : {s['input']}")
    print(f"  Answer   : {s['actual_output']}")
    print(f"  Context  : {s['context'][0][:70]}...")
    print()

Sample 1
  Question : What is the capital of France?
  Answer   : The capital of France is Paris. It is located in the north of France on the Seine river.
  Context  : Paris is the capital and largest city of France....

Sample 2
  Question : When was the Eiffel Tower built?
  Answer   : The Eiffel Tower was built in 1887 and completed in 1889 for the World's Fair.
  Context  : The Eiffel Tower was constructed between 1887 and 1889....

Sample 3
  Question : What is the population of Tokyo?
  Answer   : Tokyo has a population of approximately 14 million people in the city proper.
  Context  : Tokyo Metropolis has a population of about 14 million in its 23 specia...

Sample 4
  Question : Who wrote the Iliad?
  Answer   : The Iliad was written by Shakespeare in the 14th century.
  Context  : The Iliad is an ancient Greek epic poem attributed to Homer....



## Section 2: deepeval HallucinationMetric

The `HallucinationMetric` class scores a response against provided context.
The score is 0 (fully faithful) to 1 (hallucinated).
Set your `threshold` (the maximum hallucination score allowed to pass) based
on the risk tolerance of your use case: 0.5 for informational chatbots,
0.2 or lower for medical or financial applications.

In [5]:
# Instantiate the metric
metric = ch02.HallucinationMetric(threshold=0.5, model="gpt-4o")

print(f"Threshold : {metric.threshold}")
print(f"Model     : {metric.model}")
print(f"Using real deepeval: {ch02._DEEPEVAL_AVAILABLE}")

Threshold : 0.5
Model     : gpt-4o
Using real deepeval: False


In [6]:
# Score a single sample
sample = ch02.DEMO_SAMPLES[3]  # The Shakespeare hallucination

result = metric.score(
    input_text=sample["input"],
    actual_output=sample["actual_output"],
    context=sample["context"],
)

print(f"Question : {result.input}")
print(f"Answer   : {result.actual_output}")
print(f"Score    : {result.score:.4f}")
print(f"Passed   : {result.passed}")
print(f"Reason   : {result.reason}")

Question : Who wrote the Iliad?
Answer   : The Iliad was written by Shakespeare in the 14th century.
Score    : 0.3330
Passed   : False
Reason   : [MOCK — install deepeval==0.21.7 for real scoring]


In [7]:
# Batch score all four samples
batch_input = [
    {"input": s["input"], "actual_output": s["actual_output"], "context": s["context"]}
    for s in ch02.DEMO_SAMPLES
]

results = metric.batch_score(batch_input)

print("Batch results:")
for i, r in enumerate(results, 1):
    status = "PASS" if r.passed else "FAIL"
    print(f"  [{i}] [{status}] score={r.score:.4f}  {r.input[:55]}")

print()
stats = metric.summary_stats(results)
print("Summary stats:")
print(json.dumps(stats, indent=4))

Batch results:
  [1] [PASS] score=0.5380  What is the capital of France?
  [2] [PASS] score=0.8460  When was the Eiffel Tower built?
  [3] [PASS] score=0.7690  What is the population of Tokyo?
  [4] [FAIL] score=0.3330  Who wrote the Iliad?

Summary stats:
{
    "n": 4,
    "mean_score": 0.6215,
    "pass_rate": 0.75,
    "fail_count": 1,
    "threshold": 0.5
}


## Section 3: RAGAS faithfulness pipeline

RAGAS takes a different approach: it breaks the model's response into atomic
claims and checks each claim against the retrieved context passages.
The faithfulness score is the fraction of claims that the context supports.

This claim-level decomposition is more granular than the full-response
comparison deepeval uses, which is why the ensemble outperforms either alone.

In [8]:
# Run the RAGAS faithfulness pipeline
pipeline = ch02.RAGASPipeline(metrics=["faithfulness"])

ragas_result = pipeline.evaluate(
    questions=[s["input"] for s in ch02.DEMO_SAMPLES],
    answers=[s["actual_output"] for s in ch02.DEMO_SAMPLES],
    contexts=[s["context"] for s in ch02.DEMO_SAMPLES],
    ground_truths=[s["ground_truth"] for s in ch02.DEMO_SAMPLES],
)

print(f"Faithfulness score    : {ragas_result.faithfulness_score:.4f}")
print(f"Answer relevancy      : {ragas_result.answer_relevancy_score}")
print(f"Context recall        : {ragas_result.context_recall_score}")
print(f"Samples evaluated     : {ragas_result.n_samples}")

if not ch02._RAGAS_AVAILABLE:
    print("\n[NOTE] This is a mock result. Install ragas==0.1.21 for real scoring.")

Faithfulness score    : 0.8200
Answer relevancy      : 0.75
Context recall        : None
Samples evaluated     : 4

[NOTE] This is a mock result. Install ragas==0.1.21 for real scoring.


## Section 4: CombinedHallucinationScorer

The ensemble scorer combines deepeval and RAGAS into a single score.
Default weights: 60% deepeval, 40% RAGAS.
Adjust weights based on your validation study, whichever metric better
correlates with human judgements on your data domain should get higher weight.

In [9]:
scorer = ch02.CombinedHallucinationScorer(
    deepeval_weight=0.6,
    ragas_weight=0.4,
    threshold=0.7,
    deepeval_model="gpt-4o",
)

print(f"Weights    : deepeval={scorer.weights[0]}, ragas={scorer.weights[1]}")
print(f"Threshold  : {scorer.threshold}")

Weights    : deepeval=0.6, ragas=0.4
Threshold  : 0.7


In [10]:
# Score the Shakespeare hallucination with the ensemble
sample = ch02.DEMO_SAMPLES[3]

combined = scorer.score(
    question=sample["input"],
    answer=sample["actual_output"],
    context=sample["context"],
    ground_truth=sample["ground_truth"],
)

print(f"deepeval score     : {combined.deepeval_score:.4f}")
print(f"RAGAS faithfulness : {combined.ragas_faithfulness:.4f}")
print(f"Combined score     : {combined.combined_score:.4f}")
print(f"Passed (>={combined.threshold})   : {combined.passed}")
print(f"Weights used       : {combined.weights}")

deepeval score     : 0.3330
RAGAS faithfulness : 0.8200
Combined score     : 0.5278
Passed (>=0.7)   : False
Weights used       : (0.6, 0.4)


In [11]:
# Score all four samples with the ensemble
print("Ensemble scores for all demo samples:")
print("-" * 65)

for i, sample in enumerate(ch02.DEMO_SAMPLES, 1):
    c = scorer.score(
        question=sample["input"],
        answer=sample["actual_output"],
        context=sample["context"],
        ground_truth=sample["ground_truth"],
    )
    status = "PASS" if c.passed else "FAIL"
    print(f"  [{i}] [{status}]  de={c.deepeval_score:.3f}  ragas={c.ragas_faithfulness:.3f}  combined={c.combined_score:.3f}")
    print(f"        {sample['input'][:60]}")
    print()

Ensemble scores for all demo samples:
-----------------------------------------------------------------
  [1] [FAIL]  de=0.538  ragas=0.820  combined=0.651
        What is the capital of France?

  [2] [PASS]  de=0.846  ragas=0.820  combined=0.836
        When was the Eiffel Tower built?

  [3] [PASS]  de=0.769  ragas=0.820  combined=0.789
        What is the population of Tokyo?

  [4] [FAIL]  de=0.333  ragas=0.820  combined=0.528
        Who wrote the Iliad?



## Section 5: Cohen's kappa, validating the metric before using it as a CI gate

Before trusting any automated metric to gate a CI build, validate that it
agrees with human judgements. Cohen's kappa measures agreement beyond chance.

**Interpretation guide:**
- kappa < 0.41: too unreliable, collect more data, re-examine the metric
- 0.41–0.60: moderate, acceptable for research, not for blocking deployments
- 0.61–0.80: substantial, minimum bar for a CI gate
- 0.81–1.00: almost perfect, high confidence in automated decisions

In [12]:
# Simulated binary labels from two human raters
# 0 = faithful response, 1 = hallucinated

human_a = [0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1]
human_b = [0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1]

kappa_result = ch02.compute_cohens_kappa(human_a, human_b)

print("Human A vs Human B inter-rater reliability:")
print(f"  Kappa            : {kappa_result['kappa']:.4f}")
print(f"  Interpretation   : {kappa_result['interpretation']}")
print(f"  OK for CI gate   : {kappa_result['acceptable_for_ci_gate']}")
print(f"  N samples        : {kappa_result['n']}")

Human A vs Human B inter-rater reliability:
  Kappa            : 0.6939
  Interpretation   : Substantial
  OK for CI gate   : True
  N samples        : 20


In [13]:
# Model (automated metric) vs Human A
# This tests whether your automated metric can replace human review in CI

model_labels = [0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1]

model_kappa = ch02.compute_cohens_kappa(human_a, model_labels)

print("Automated metric vs Human A:")
print(f"  Kappa            : {model_kappa['kappa']:.4f}")
print(f"  Interpretation   : {model_kappa['interpretation']}")
print(f"  OK for CI gate   : {model_kappa['acceptable_for_ci_gate']}")

if model_kappa["acceptable_for_ci_gate"]:
    print("\n  Conclusion: Automated metric agreement is substantial.")
    print("  Safe to use as a CI gate.")
else:
    print("\n  Conclusion: Agreement is too low for automated gating.")
    print("  Increase human validation sample before promoting to CI.")

Automated metric vs Human A:
  Kappa            : 0.8980
  Interpretation   : Almost perfect
  OK for CI gate   : True

  Conclusion: Automated metric agreement is substantial.
  Safe to use as a CI gate.


## Section 6: Power analysis, how many samples do I need?

Before running a hallucination evaluation study, compute the sample size
needed to detect a meaningful change with statistical confidence.

This prevents two expensive mistakes:
- **Too few samples**: you miss real regressions (false negatives)
- **Too many samples**: you over-invest in evaluation when a smaller set would suffice

The two-proportion z-test is the correct model here: you are comparing
the hallucination rate of the production model vs the candidate model.

In [14]:
# Scenario: current system hallucinates 15% of the time
# You want to detect a 5 percentage point improvement

result = ch02.compute_sample_size(
    baseline_rate=0.15,
    minimum_detectable_effect=0.05,
    alpha=0.05,
    power=0.80,
)

import json
print(json.dumps(result, indent=2))

{
  "baseline_rate": 0.15,
  "alternative_rate": 0.2,
  "minimum_detectable_effect": 0.05,
  "alpha": 0.05,
  "power": 0.8,
  "cohens_h": 0.1319,
  "n_per_group": 452,
  "n_total": 904,
  "interpretation": "You need 452 samples per model to detect a 5% shift in hallucination rate with 80% power at alpha=0.05."
}


In [15]:
# Compare sample requirements across different scenarios

scenarios = [
    (0.15, 0.03, "Detect 3pp improvement from 15% baseline"),
    (0.15, 0.05, "Detect 5pp improvement from 15% baseline"),
    (0.15, 0.10, "Detect 10pp improvement from 15% baseline"),
    (0.30, 0.05, "Detect 5pp improvement from 30% baseline"),
    (0.05, 0.02, "Detect 2pp improvement from 5% baseline (high-quality system)"),
]

print(f"{'Scenario':<50} {'n/group':>8} {'n total':>8}")
print("-" * 68)

for baseline, mde, description in scenarios:
    r = ch02.compute_sample_size(baseline, mde)
    print(f"{description:<50} {r['n_per_group']:>8} {r['n_total']:>8}")

Scenario                                            n/group  n total
--------------------------------------------------------------------
Detect 3pp improvement from 15% baseline               1200     2400
Detect 5pp improvement from 15% baseline                452      904
Detect 10pp improvement from 15% baseline               124      248
Detect 5pp improvement from 30% baseline                688     1376
Detect 2pp improvement from 5% baseline (high-quality system)     1100     2200


## Section 7: Visualizing metric distributions

Plotting the score distribution helps identify threshold calibration issues.
A bimodal distribution (many scores near 0, many near 1) suggests the metric
is making confident decisions. A unimodal distribution near 0.5 suggests
the metric is uncertain and the threshold needs validation.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import random

    random.seed(42)

    # Simulate score distributions for two models
    # Production model: mostly faithful with some errors
    prod_scores = ([random.uniform(0.7, 1.0) for _ in range(70)] +
                   [random.uniform(0.0, 0.4) for _ in range(30)])

    # Candidate model: improved but not perfect
    cand_scores = ([random.uniform(0.75, 1.0) for _ in range(80)] +
                   [random.uniform(0.0, 0.4) for _ in range(20)])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

    axes[0].hist(prod_scores, bins=20, color="#e74c3c", alpha=0.8, edgecolor="white")
    axes[0].axvline(x=0.80, color="black", linestyle="--", label="Threshold (0.80)")
    axes[0].set_title("Production Model Score Distribution", fontweight="bold")
    axes[0].set_xlabel("Faithfulness Score")
    axes[0].set_ylabel("Count")
    axes[0].legend()

    axes[1].hist(cand_scores, bins=20, color="#2ecc71", alpha=0.8, edgecolor="white")
    axes[1].axvline(x=0.80, color="black", linestyle="--", label="Threshold (0.80)")
    axes[1].set_title("Candidate Model Score Distribution", fontweight="bold")
    axes[1].set_xlabel("Faithfulness Score")
    axes[1].legend()

    prod_pass_rate = sum(1 for s in prod_scores if s >= 0.80) / len(prod_scores)
    cand_pass_rate = sum(1 for s in cand_scores if s >= 0.80) / len(cand_scores)

    fig.suptitle(
        f"Production pass rate: {prod_pass_rate:.0%}   Candidate pass rate: {cand_pass_rate:.0%}",
        fontsize=12
    )

    plt.tight_layout()
    plt.savefig("ch02_score_distributions.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Chart saved to ch02_score_distributions.png")

except ImportError:
    print("matplotlib not installed — skipping chart. pip install matplotlib to enable.")

## Summary

This notebook built the complete hallucination detection stack from Chapter 2.

### What you built
- A `HallucinationMetric` wrapper around deepeval with mock fallback for dry runs
- A `RAGASPipeline` for claim-level faithfulness scoring
- A `CombinedHallucinationScorer` ensemble (60% deepeval, 40% RAGAS)
- Cohen's kappa computation to validate metric agreement with human judgements
- Power analysis to plan the right evaluation sample size

### Next steps
1. **Chapter 3**, Wire the `CombinedHallucinationScorer` into a CI gate using `HallucinationGate`, then add it to GitHub Actions.
2. **Validate with your data**, Run the kappa analysis on 50+ human-labeled examples from your actual system before using the metric as a CI gate.
3. **Calibrate the threshold**, Use the power analysis to decide how many test samples you need for each deployment.
4. **Tune ensemble weights**, If one metric consistently disagrees with human labels on your domain, reduce its weight.
5. **Build a regression suite**, Save every case where the metric flags a failure. Run it on every model update.